# Clone Repository and set up the Environment

In [2]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available")


Braindecode version: 1.3.0
CTNet is available


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # or ":16:8" if memory is tight
os.environ["PYTHONHASHSEED"] = "0"                  # optional, extra stability


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [12]:
from datetime import datetime
from collections import defaultdict

from torch.utils.data import Subset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, CosineAnnealingWarmRestarts

from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion


from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint

#EEGMamba-MOE approximation
from models.eeg_mamba_fft import EEGMamba


# Loading data for training

In [ ]:
#import numpy as np
from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
from braindecode.preprocessing import create_windows_from_events
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset


device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = MOABBDataset(
    dataset_name='BNCI2014001', subject_ids=[1]
)

low_cut_hz = 4.0
high_cut_hz = 38.0
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000


pre_norms = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),
    Preprocessor(lambda data, factor: np.multiply(data, factor), factor=1e6),  # V→µV
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),
]
pre_ems = [
    Preprocessor(exponential_moving_standardize, factor_new=factor_new, init_block_size=init_block_size),
]

preprocess(dataset, pre_norms, n_jobs=-1)


trial_start_offset_seconds = -0.5

sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])

trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)


windows_pre = create_windows_from_events(
    dataset,
    trial_start_offset_samples = int(-0.5 * sfreq) ,
    trial_stop_offset_samples = 0,
    preload=True,
    # verbose=0
)


# Split into train and test
splitted = windows_pre.split("session")
train_set_pre = splitted["0train"]  # Session train
test_set_pre = splitted["1test"]  # Session evaluation


X_train_pre = np.stack([train_set_pre[i][0] for i in range(len(train_set_pre))])
train_mean_pre = X_train_pre.mean(axis=(0, 2), keepdims=True)
train_std_pre  = X_train_pre.std(axis=(0, 2), keepdims=True) + 1e-6
train_min_pre  = X_train_pre.min(axis=(0, 2), keepdims=True)
train_max_pre  = X_train_pre.max(axis=(0, 2), keepdims=True)

# Save prenorm stats + channel names for later mapping/clamping
ch_names_pre = windows_pre.datasets[0].raw.info["ch_names"]
os.makedirs("results", exist_ok=True)
np.savez(f"results/S{1}_prenorm_stats.npz",  
         mean=train_mean_pre, std=train_std_pre,
         vmin=train_min_pre, vmax=train_max_pre,
         ch_names=np.array(ch_names_pre))


preprocess(dataset, pre_ems, n_jobs=-1)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=int(-0.5 * sfreq),
    trial_stop_offset_samples=0,
    preload=True,
)


splitted = windows_dataset.split("session")
train_set = splitted["0train"]
test_set  = splitted["1test"]


# Build simple tensors to compute stats on train windows only
X_train = SliceDataset(train_set, idx=0)

y_train = np.array([y for y in SliceDataset(train_set, idx=1)])

X_test = np.stack([test_set[i][0] for i in range(len(test_set))])
y_test = np.array(test_set.get_metadata().target)

train_indices, val_indices = train_test_split(
      X_train.indices_, test_size=0.2, shuffle=False
  )
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)

X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T


X = torch.tensor(X_test, dtype=torch.float32, device=device)
y = torch.tensor(y_test, dtype=torch.long, device=device)

x, y , meta = train_set[0]
print(type(x), x.shape, x.mean(), x.std())


Check if this loads as train_set = x, y or x, y , meta

---
Useful for later

In [ ]:
sample = train_set[0]
print(type(sample))
print(len(sample))   # see what fields exist ()
channel_names = dataset.datasets[0].raw.info['ch_names']
print(channel_names)

## Set loading functions

In [ ]:
from utils import load_subject as ls

## Data loading sanity check

Use subject 1 as this is the standard subject for training

In [ ]:
# Run once or sanity check
subject_id = 1
train_set, test_set, train_subset, val_subset, adv = ls.load_subject_data_cached("BNCI2014001", subject_id)
train_mean_pre, train_std_pre, train_min_pre, train_max_pre = adv


print(f"Window shape: {windows_dataset[0][0].shape}")
train_subset[0][0].shape[0]
channel_names = test_set.datasets[0].raw.info['ch_names']
print(channel_names)
print(train_mean_pre)
print(train_std_pre)
print(train_min_pre)
print(train_max_pre)

## Load and inspect model

In [ ]:
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()
device = "cuda" if cuda else "cpu"

seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
)

if cuda:
    torch.backends.cudnn.benchmark = True
    model.cuda()

# Training

## Set model hyper params

In [ ]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}
alt_params = {'lr': 1e-3, 'batch_size': 128, 'weight_decay': 5e-3, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

## Single run mode

Single training run for model debugging, sanity checks and testing non braindecode architectures (EEGMammba)



In [ ]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

train_set, test_set, train_subset, val_subset, adv = ls.load_subject_data_cached("BNCI2014001", 1)

transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),

]


# Extract model params from dataset, initialise model and set hyper-parameters
classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
n_classes = len(classes)
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
)

# Hyper params
params = mamba_params

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def make_scheduler(optimizer, last_epoch=-1):
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=10, last_epoch=last_epoch)
    cosine = CosineAnnealingLR(optimizer, T_max=n_epochs-10, last_epoch=last_epoch)
    return CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    
    criterion=torch.nn.CrossEntropyLoss,

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],

    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=["accuracy",],
    device=device,
    classes=classes,
    max_epochs=500,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")



# Record Baselines for chosen models

In [ ]:
from utils.train_helpers import train_single_run, create_baseline_table

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Baseline Run

seeds = [42, 123, 2024, 31415, 999]   

datasets = {
    "BNCIv2": ("BNCI2014001", 9),
    # Ended up not using DEAP. Can still use for further experiments 
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_CONFIGS = {
    'EEGNet': {
        'model_class': EEGNetv4,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'DeepConvNet': {
        'model_class': Deep4Net,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'CTNet': {
        'model_class': CTNet,
        'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    }
}

# Store all results: results[model_name][subject_id] = [acc1, acc2, acc3, acc4, acc5]
all_results = defaultdict(lambda: defaultdict(list))

for model_name in MODEL_CONFIGS.keys():
    print(f"RUNNING BASELINE FOR {model_name.upper()}")
    print(f"{'='*60}")
    
    config = MODEL_CONFIGS[model_name]
    
    for subject_id in subjects:
        print(f"\n--- Subject {subject_id} ---")

        subject_scores = []
        for seed in seeds:
            print(f"  Seed {seed}: RUNNING")
            try:
                accuracy = train_single_run(model_name, subject_id, seed, dataset, config, device)
                subject_scores.append(accuracy)
                print(f"  Seed {seed}: {accuracy:.4f}")
            except Exception as e:
                print(f"  Seed {seed}: FAILED ({e})")
                subject_scores.append(np.nan)

        # Store results for this (model, subject) pair
        all_results[model_name][subject_id] = subject_scores

        # Calculate stats for this subject
        valid_scores = [s for s in subject_scores if not np.isnan(s)]
        if valid_scores:
            mean_acc = np.mean(valid_scores)
            std_acc = np.std(valid_scores)
            print(f"  Subject {subject_id} baseline: {mean_acc:.4f} ± {std_acc:.4f}")
        else:
            print(f"  Subject {subject_id}: ALL RUNS FAILED")

# Create and display results
baseline_df = create_baseline_table(all_results, subjects)
print(f"\n{'='*80}")
print("FINAL BASELINE RESULTS")
print(f"{'='*80}")

# Subject-wise baselines
for model_name in MODEL_CONFIGS.keys():
    print(f"\n{model_name}:")
    model_data = baseline_df[baseline_df['Model'] == model_name]

    subject_means = []
    for _, row in model_data.iterrows():
        if not np.isnan(row['Mean_Accuracy']):
            print(f"  Subject {row['Subject']}: {row['Mean_Accuracy']:.4f} ± {row['Std_Accuracy']:.4f}")
            subject_means.append(row['Mean_Accuracy'])
        else:
            print(f"  Subject {row['Subject']}: FAILED")

    # Dataset-wide average
    if subject_means:
        dataset_mean = np.mean(subject_means)
        dataset_std = np.std(subject_means)
        print(f"  → Dataset average: {dataset_mean:.4f} ± {dataset_std:.4f}")
    else:
        print(f"  → Dataset average: FAILED")

# Save results
baseline_df.to_csv('baseline_results.csv', index=False)
print(f"\nResults saved to baseline_results.csv")

# Adversarial attacks


## Initialize model and load checkpoints

In [ ]:
# Sanity Check that the function works as intended
from utils.load_adv_data import load_adv_test_data

X, y, train_min_t, train_max_t, train_std_np, train_mean_np, CH_NAMES = load_adv_test_data(1)

print(X.shape)
print(y.shape)
print(train_min_t.shape)
print(train_max_t.shape)
print(train_std_np.shape)

n_channels = int(X.shape[1])
n_times    = int(X.shape[2])
n_classes  = int(y.max().item() + 1)  
print(n_channels)
print(n_times)
print(n_classes)



In [ ]:

from braindecode.util import set_random_seeds

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
cuda = torch.cuda.is_available()

if device == "cuda":
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

# Print original CTNet keys
path = torch.load('/content/robust-eeg-models/results/CTNet/CTNet_S1_seed123/checkpoint.pth')
print(path.keys())
model.load_state_dict(path['state_dict'])   # <- no .eval() here
model.eval()


X, y, train_min_t, train_max_t, train_std_np, train_mean_np, CH_NAMES  = load_adv_test_data(1)


print(CH_NAMES)
print(len(CH_NAMES))

## Initialise variables

## Set up, helper methods for adversarial attacks and eval mode

In [ ]:
from attack import attack_metrics as am

In [16]:
from attack.attack_explainers import AttackExplainers

In [ ]:
# ========= Cell 3: Subject setup + full run =========
import os, json, torch, pandas as pd
import captum.attr as CA
from repr import repr_helpers as rp
from utils.load_adv_data import load_adv_test_data


SEEDS = [42, 123, 2024, 31415, 999]

SUBJECT_ID = 9  #Subjects 1, 3, 8 & 9 used 

# ---- model builders and checkpoints ----
MODEL_BUILDERS = {
    "EEGNet":      lambda: EEGNetv4(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "DeepConvNet": lambda: Deep4Net(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "CTNet":       lambda: CTNet(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "Mamba":       lambda: EEGMamba(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
}

CKPTS = {
    "EEGNet":       {s: f"results/EEGNet/EEGNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"          for s in SEEDS},
    "DeepConvNet":  {s: f"results/DeepConvNet/DeepConvNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth" for s in SEEDS},
    "CTNet":        {s: f"results/CTNet/CTNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"             for s in SEEDS},
    "Mamba":        {s: f"results/EEGMamba/EEGMamba_S{SUBJECT_ID}_seed{s}/checkpoint.pth"       for s in SEEDS},
}


device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

muV_grid = [0.25, 0.5, 1.0, 2.0]  
    
data = load_adv_test_data(SUBJECT_ID)
if isinstance(data, (list, tuple)) and len(data) >= 8:
    X, y, train_min_t, train_max_t, train_std_np, CH_NAMES, train_mean_np, prenorm_std_np = data[:8]
else:
    X, y, train_min_t, train_max_t, train_std_np, train_mean_np, CH_NAMES = data
    # Fallback if prenorm stds not provided by loader (use EMS stds as proxy)
    prenorm_std_np = train_std_np

# map μV -> εᶻ uses PRENORM stats
median_std_pre = float(np.median(prenorm_std_np))

# ROI indices for this subject’s channels
ROI_IDX = [CH_NAMES.index(n) for n in ("C3","C4","Cz") if n in CH_NAMES]


# move data to device
X = X.to(device); y = y.to(device)
train_min_t = train_min_t.to(device); train_max_t = train_max_t.to(device)

ATTR_MAX_N = 128
N_STEPS_IG = 16
IG_INT_BS  = 8
    

def run_for_model_seed(model_name: str, seed: int):

    rp.set_all_seeds(seed)

    # build & load
    m = MODEL_BUILDERS[model_name]().to(device)
    ckpt_path = CKPTS[model_name][seed]
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
    m.load_state_dict(state_dict)
    m.eval()
    model = m
    ig = CA.IntegratedGradients(model)

    explainers = AttackExplainers(
        ATTR_MAX_N=ATTR_MAX_N,
        N_STEPS_IG=N_STEPS_IG,
        ig=ig,
        muV_grid=muV_grid,
        model=model,
        X=X,
        y=y,
        train_min_t=train_min_t,
        train_max_t=train_max_t,
        prenorm_std_np=prenorm_std_np,
        median_std_pre=median_std_pre,
        CH_NAMES=None
    )

    # fixed clean subset for attributions

    IDX_CAP = slice(0, min(ATTR_MAX_N, X.size(0)))
    X_cap_clean = X[IDX_CAP].detach().clone()
    y_cap       = y[IDX_CAP].detach().clone()

    # clean explanations cache
    E_CLEAN = {name: fn(X_cap_clean, y_cap) for name, fn in explainers.EXPLAINERS.items()}

    rows = []
    # FGSM / PGD standard
    rows += explainers.run_linf_sweep("FGSM", muV_grid, steps=None, lp_sigma_t=None, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)
    rows += explainers.run_linf_sweep("PGD",  muV_grid, steps=40,    alpha_rule=lambda e: e/8, lp_sigma_t=None, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # low-pass (LP) variants (σᵗ=3.0)
    rows += explainers.run_linf_sweep("FGSM", muV_grid, steps=None, lp_sigma_t=3.0, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)
    rows += explainers.run_linf_sweep("PGD",  muV_grid, steps=40,    alpha_rule=lambda e: e/8, lp_sigma_t=3.0, cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # DeepFool (L2)
    rows += explainers.run_deepfool(cap_idx=IDX_CAP, E_CLEAN=E_CLEAN)

    # tag rows
    for r in rows:
        r["subject_id"] = SUBJECT_ID
        r["model_name"] = model_name
        r["seed"] = seed
    return rows



n_channels = int(X.shape[1])
n_times    = int(X.shape[2])
n_classes  = int(y.max().item() + 1)
print(f"Data shape: {X.shape} | n_ch={n_channels}, n_times={n_times}, n_classes={n_classes}")
print(f"Median PRENORM std (μV): {median_std_pre:.6f} | μV grid: {muV_grid}")

all_rows = []
for model_name in MODEL_BUILDERS.keys():
    print(f"Running: {model_name}")
    for seed in SEEDS:
        all_rows.extend(run_for_model_seed(model_name, seed))

# ---- save per-subject CSV ----
os.makedirs("results", exist_ok=True)
csv_path = f"results/adversarial_results_{SUBJECT_ID}.csv"
pd.DataFrame(all_rows).to_csv(csv_path, index=False)
print(f"Wrote {csv_path} with {len(all_rows)} rows.")

# ---- update master ----
master_path = "results/adversarial_results_MASTER.csv"
if os.path.isfile(master_path):
    pd.concat([pd.read_csv(master_path), pd.DataFrame(all_rows)], ignore_index=True).to_csv(master_path, index=False)
else:
    pd.DataFrame(all_rows).to_csv(master_path, index=False)
print(f"Updated {master_path}.")


Data shape: torch.Size([288, 22, 1125]) | n_ch=22, n_times=1125, n_classes=4
Median PRENORM std (μV): 11.271589 | μV grid: [0.25, 0.5, 1.0, 2.0]
Running: EEGNet


<decorator-gen-492>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

# Analysis
## Load the master results containing all data

## Clean the data

In [22]:
from analysis import clean_data as cd

df_raw = pd.read_csv('/Users/temp/Documents/Bath University/Msc/Disseration /EEG_Project/results/adversarial_results_MASTER.csv')
df_clean_master = cd.clean_adversarial_data(df_raw)


Initial dataset shape: (1360, 40)
Missing values in spearman_IG: 0
Missing values in clean_acc: 0
Dropped 0 rows with missing interpretability data
Found 0 potential training failures (clean_acc < 0.3)
Impossible ASR values: 0
Impossible accuracy values: 0
Impossible Spearman values: 0
ASR consistency violations: 0
Extreme SNR outliers (3*IQR): 27

Final dataset summary:
Final shape: (1360, 44)
Models: {'EEGNet': 340, 'DeepConvNet': 340, 'CTNet': 340, 'Mamba': 340}
Subjects: 4
Attack types: {'FGSM': 320, 'PGD': 320, 'FGSM_LP': 320, 'PGD_LP': 320, 'DeepFool_L2': 80}
Seeds per condition: count    80.000000
mean     17.000000
std       6.037855
min       5.000000
25%      20.000000
50%      20.000000
75%      20.000000
max      20.000000
dtype: float64

Remaining missing values:
smooth                       80
smooth_type                 720
smooth_sigma_t              720
muV_budget                   80
eps_z                        80
eps_uV_per_channel           80
steps                

In [ ]:
df_clean_master = pd.read_csv('/Users/temp/Documents/Bath University/Msc/Disseration /EEG_Project/adversarial_results_MODIFIED_clean1.csv')

## Drop Irrelevant columns

In [ ]:
columns_to_drop = ['eps_z', 'random_start','adv_acc','steps', 'alpha', 'smooth' ,'frac_at_boundary', 'restarts', 'smooth_sigma_t', 'smooth_type', 'targeted', 'top5_roi_share_clean_IG', 'top5_roi_share_adv_IG','top5_roi_share_delta_IG' ]

df_clean_all = df_clean_master.drop(columns=columns_to_drop)


In [ ]:
df_clean_all.info()


In [ ]:
from analysis import attack_summary as ats

df, successful_df, threshold = ats.load_and_prepare_data('/Users/temp/Documents/Bath University/Msc/Disseration /EEG_Project/results/adversarial_results_MASTER.csv')
summary = ats.calculate_attack_summary(successful_df)

# Create individual plots
easr_fig = ats.plot_easr_comparison(summary)
spearman_fig = ats.plot_spearman_comparison(summary)
scatter_fig = ats.plot_asr_vs_stability_scatter(summary)
arch_fig = ats.plot_architecture_comparison(successful_df, threshold)
detailed_fig = ats.plot_deepfool_detailed_analysis(successful_df)

# Save plots
easr_fig.savefig('training/figures/easr_comparison.png', dpi=300, bbox_inches='tight')
spearman_fig.savefig('training/figures/spearman_comparison.png', dpi=300, bbox_inches='tight')
scatter_fig.savefig('training/figures/asr_vs_stability.png', dpi=300, bbox_inches='tight')
arch_fig.savefig('training/figures/architecture_heatmap.png', dpi=300, bbox_inches='tight')
detailed_fig.savefig('training/figures/deepfool_detailed_analysis.png', dpi=300, bbox_inches='tight')

plt.show()


## Aggregate accross seeds

In [ ]:
import pandas as pd
import numpy as np

def aggregate_across_seeds(df_clean):
    """
    Simple aggregation across seeds for experimental conditions
    """
    print("SEED AGGREGATION")
    print("=" * 50)
    print(f"Input dataset shape: {df_clean.shape}")

    # Define grouping variables
    grouping_vars = ['subject_id', 'model_name', 'attack', 'architecture_type', 'muV_budget']

    # For numeric columns, calculate mean and std
    numeric_vars = [
        'ASR', 'spearman_IG', 'clean_acc', 'snr_db_mean', 'snr_db_std',
        'median_L2_success', 'mean_L2_all', 'acc_drop', 'rel_acc_drop'
    ]

    # Filter to only include columns that actually exist in the dataframe
    numeric_vars = [var for var in numeric_vars if var in df_clean.columns]

    # Perform aggregation
    df_aggregated = df_clean.groupby(grouping_vars).agg(
        **{var: (var, 'mean') for var in numeric_vars},  # Original names for means
        **{f'{var}_std': (var, 'std') for var in numeric_vars},
        **{f'{var}_count': (var, 'count') for var in numeric_vars},
        n_seeds=('seed', 'nunique')
    ).reset_index()

    print(f"Output dataset shape: {df_aggregated.shape}")

    return df_aggregated

# Usage
df_aggregated = aggregate_across_seeds(df_clean_all)

In [ ]:
from analysis import statistical_analysis as sa

# Usage:
df_analysis = sa.statistical_analysis(df_aggregated)


In [ ]:
from analysis import visualizations as vis
# Usage: =
fig1, fig2 = vis.create_visualizations(df_analysis)

# Calculating the explanation Blind Spot

In [ ]:
# Define a threshold for explanation similarity
explanation_threshold = 0.001

df = df_aggregated.copy()
# Create a new binary column: 1 if the explanation was fooled, 0 otherwise
df['explanation_fooled'] = (df['spearman_IG'] < explanation_threshold).astype(int)

In [ ]:
# Filter the dataset to only include successful attacks
successful_attacks_df = df[df['ASR'] == 1]

# Calculate the Explanation ASR
# This is the mean of the 'explanation_fooled' column for successful attacks
explanation_asr = successful_attacks_df['explanation_fooled'].mean()

print(f"Explanation Attack Success Rate (E-ASR): {explanation_asr:.2%}")
print(f"This means {explanation_asr:.2%} of successful attacks also significantly altered the explanation.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Calculate median spearman_IG for all data and for successful attacks
median_spearman_all = df['spearman_IG'].median()
median_spearman_success = df[df['ASR'] == 1]['spearman_IG'].median()

print(f"Median spearman_IG for all samples: {median_spearman_all:.4f}")
print(f"Median spearman_IG for successful attacks: {median_spearman_success:.4f}")

In [ ]:
# Set threshold based on your data - here, using median of successful attacks
explanation_threshold = median_spearman_success  # or use a fixed value like 0.3 if preferred

# Create a binary column for explanation fooled
df['explanation_fooled'] = (df['spearman_IG'] < explanation_threshold).astype(int)

# Calculate overall Explanation Attack Success Rate (E-ASR)
successful_attacks_df = df[df['ASR'] == 1]
explanation_asr = successful_attacks_df['explanation_fooled'].mean()
print(f"\nExplanation Attack Success Rate (E-ASR) at threshold {explanation_threshold:.4f}: {explanation_asr:.2%}")

In [ ]:
# Group by architecture
e_asr_by_architecture = successful_attacks_df.groupby('architecture_type')['explanation_fooled'].mean()
print("\nE-ASR by Architecture:")
print(e_asr_by_architecture)

# Group by attack type
e_asr_by_attack = successful_attacks_df.groupby('attack')['explanation_fooled'].mean()
print("\nE-ASR by Attack Type:")
print(e_asr_by_attack)

# Group by both architecture and attack
e_asr_by_arch_attack = successful_attacks_df.groupby(['architecture_type', 'attack'])['explanation_fooled'].mean().reset_index()
print("\nE-ASR by Architecture and Attack:")
print(e_asr_by_arch_attack)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=e_asr_by_arch_attack, x='architecture_type', y='explanation_fooled', hue='attack')
plt.title(f'Explanation Attack Success Rate (E-ASR) at Threshold {explanation_threshold:.2f}')
plt.ylabel('E-ASR (Probability Explanation is Fooled | Attack Successful)')
plt.ylim(0, 1)
plt.legend(title='Attack Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()